# Llama 3.2 1B Instruct → ExecuTorch `.pte` (Google Colab)

This notebook makes the **Llama file the app needs**: `llama3_2_1b.pte` + `tokenizer.model`.
One Llama file serves **both** detectors. The SMS detector and the network detector each send their verdict to the same Llama, which writes the explanation. You don't change the two trained models at all.

**Before you start**
- Your Hugging Face account must have access to `meta-llama/Llama-3.2-1B-Instruct` (✅ you have it).
- Create a token: huggingface.co → Settings → Access Tokens → **New token**, type **Read**. Copy it.
- Colab: Runtime → Change runtime type → CPU is fine. A **High-RAM** runtime is safer if you have it. Export needs a lot of memory.

Runtime → Run all. Expect roughly 15–30 minutes (most of it is the export).

## 1. Install

In [6]:
!pip install -q executorch pytorch-tokenizers huggingface_hub safetensors
import psutil, executorch, torch
print("executorch", executorch.version.__version__ if hasattr(executorch, "version") else "", "| torch", torch.__version__)
print(f"RAM: {psutil.virtual_memory().total/1e9:.1f} GB total, {psutil.virtual_memory().available/1e9:.1f} GB free")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
executorch  | torch 2.15.0+cu132
RAM: 13.6 GB total, 11.8 GB free


## 2. Log in to Hugging Face
Paste your **Read** token when asked. It stays in this Colab session only. Don't put it in the notebook text or on GitHub.
(Optional: add it under the 🔑 *Secrets* tab as `HF_TOKEN` to skip the prompt.)

In [8]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    login()          # shows a box: paste your token

## 3. Download Llama 3.2 1B Instruct (Meta format, ~2.5 GB)
We download only the `original/` folder: `consolidated.00.pth` (weights), `params.json` (architecture), `tokenizer.model` (tokenizer). This is the format ExecuTorch's exporter expects.

**Plan B (no Meta access):** set `USE_UNGATED_MIRROR = True`. It downloads the community copy `unsloth/Llama-3.2-1B-Instruct` (same weights, Hugging Face format) and converts it. We tested the converter on a small stand-in model and its outputs matched the original to 0.0000002. With that mirror, the tokenizer comes as `tokenizer.json` instead of `tokenizer.model`.

In [9]:
import os, json, shutil
from huggingface_hub import snapshot_download
USE_UNGATED_MIRROR = False
SRC = "llama-3.2-1b-instruct/original"

if not USE_UNGATED_MIRROR:
    snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"], local_dir="llama-3.2-1b-instruct")
    TOKENIZER = f"{SRC}/tokenizer.model"
else:
    import torch
    from safetensors.torch import load_file
    hf_dir = snapshot_download("unsloth/Llama-3.2-1B-Instruct", local_dir="hf-llama")
    cfg = json.load(open(f"{hf_dir}/config.json")); sd = load_file(f"{hf_dir}/model.safetensors")
    dim, nh, nkv = cfg["hidden_size"], cfg["num_attention_heads"], cfg["num_key_value_heads"]
    hd = cfg.get("head_dim") or dim // nh
    def unpermute(w, n):   # undo Hugging Face's rotary-embedding row order
        return w.view(n, 2, hd // 2, dim).transpose(1, 2).reshape(n * hd, dim)
    meta = {"tok_embeddings.weight": sd["model.embed_tokens.weight"], "norm.weight": sd["model.norm.weight"],
            "output.weight": sd.get("lm_head.weight", sd["model.embed_tokens.weight"])}
    for i in range(cfg["num_hidden_layers"]):
        p = f"model.layers.{i}."
        meta.update({f"layers.{i}.attention.wq.weight": unpermute(sd[p+"self_attn.q_proj.weight"], nh),
                     f"layers.{i}.attention.wk.weight": unpermute(sd[p+"self_attn.k_proj.weight"], nkv),
                     f"layers.{i}.attention.wv.weight": sd[p+"self_attn.v_proj.weight"],
                     f"layers.{i}.attention.wo.weight": sd[p+"self_attn.o_proj.weight"],
                     f"layers.{i}.feed_forward.w1.weight": sd[p+"mlp.gate_proj.weight"],
                     f"layers.{i}.feed_forward.w2.weight": sd[p+"mlp.down_proj.weight"],
                     f"layers.{i}.feed_forward.w3.weight": sd[p+"mlp.up_proj.weight"],
                     f"layers.{i}.attention_norm.weight": sd[p+"input_layernorm.weight"],
                     f"layers.{i}.ffn_norm.weight": sd[p+"post_attention_layernorm.weight"]})
    os.makedirs(SRC, exist_ok=True)
    torch.save({k: v.contiguous() for k, v in meta.items()}, f"{SRC}/consolidated.00.pth")
    rope = cfg.get("rope_scaling") or {}
    json.dump({"dim": dim, "hidden_dim": cfg["intermediate_size"], "n_heads": nh, "n_kv_heads": nkv,
               "n_layers": cfg["num_hidden_layers"], "norm_eps": cfg["rms_norm_eps"], "rope_theta": cfg["rope_theta"],
               "use_scaled_rope": rope.get("rope_type") == "llama3", "rope_scale_factor": rope.get("factor", 32),
               "vocab_size": cfg["vocab_size"]}, open(f"{SRC}/params.json", "w"))
    del sd, meta
    TOKENIZER = f"{hf_dir}/tokenizer.json"
print(os.listdir(SRC), "| tokenizer:", TOKENIZER)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

['consolidated.00.pth', 'params.json', 'tokenizer.model'] | tokenizer: llama-3.2-1b-instruct/original/tokenizer.model


## 4. Export to ExecuTorch (the important step)
These are the same settings as `scripts/export_llama_pte.sh` in the project:
- **KV cache + SDPA:** remembers earlier words, so each new word is fast.
- **8da4w quantisation:** 4-bit weights and 8-bit activations. About 4× smaller and faster on Arm CPUs.
- **XNNPACK backend:** optimised kernels for Arm (and x86) CPUs.
- **max_seq_length 1024:** our prompts are about 200 tokens, so this is plenty.

If Colab crashes here with "out of memory", switch to a High-RAM runtime, or run `scripts/export_llama_pte.sh` on a laptop with 16 GB RAM.

In [10]:
import subprocess,sys,os,json
BOS, EOS = 128000, [128009, 128001]   # Llama 3 <|begin_of_text|>, <|eot_id|>, <|end_of_text|>
SRC = "llama-3.2-1b-instruct/original"
cmd = [sys.executable , "-m", "executorch.extension.llm.export.export_llm",
       "base.model_class=llama3_2",
       f"base.checkpoint={SRC}/consolidated.00.pth",
       f"base.params={SRC}/params.json",
       "base.metadata=" + json.dumps(json.dumps({"get_bos_id": BOS, "get_eos_ids": EOS})),
       "model.use_kv_cache=True", "model.use_sdpa_with_kv_cache=True", "model.dtype_override=fp32",
       "backend.xnnpack.enabled=True", "backend.xnnpack.extended_ops=True",
       "quantization.qmode=8da4w", "quantization.group_size=128", "quantization.embedding_quantize='4,32'",
       "export.max_seq_length=1024", "export.max_context_length=1024",
       "export.output_name=llama3_2_1b.pte"]
res = subprocess.run(cmd, capture_output=True, text=True)
print("\n".join(l for l in (res.stdout + res.stderr).splitlines() if "INFO" in l or "Error" in l)[-2000:])
assert res.returncode == 0 and os.path.exists("llama3_2_1b.pte"), "Export failed - see the messages above"
print(f"llama3_2_1b.pte: {os.path.getsize('llama3_2_1b.pte')/1e6:.0f} MB")

[2026-11-24 18:21:21,811][root][INFO] - Applying quantizers: []
[2026-11-24 18:21:22,338][root][INFO] - Checkpoint dtype: torch.bfloat16
[2026-11-24 18:27:50,379][root][INFO] - Looking for libcustom_ops_aot_lib.so in /usr/local/lib/python3.13/dist-packages/executorch/extension/llm/custom_ops
[2026-11-24 18:27:50,381][root][INFO] - Loading custom ops library: /usr/local/lib/python3.13/dist-packages/executorch/extension/llm/custom_ops/libcustom_ops_aot_lib.so
[2026-11-24 18:27:50,434][root][INFO] - Replacing KVCache with CustomKVCache. This modifies the model in place.
[2026-11-24 18:27:50,570][root][INFO] - Model after source transforms: Transformer(
[2026-11-24 18:27:50,570][root][INFO] - Exporting with:
[2026-11-24 18:27:50,573][root][INFO] - inputs: (tensor([[2, 3, 4]]), {'input_pos': tensor([0])})
[2026-11-24 18:27:50,573][root][INFO] - kwargs: None
[2026-11-24 18:27:50,574][root][INFO] - dynamic shapes: ({1: Dim('token_dim', min=0, max=1023)}, {'input_pos': {0: 1}})
[2026-11-24 18:

## 5. Test it with both detectors' prompts
These are the same prompts the app builds (`src/explainer.py` for SMS, `src/network_detector.py` for network flows). Llama runs here through ExecuTorch's `TextLLMRunner`, exactly as in the app.

In [11]:
import time
import executorch.extension.llm.custom_ops.custom_ops   # registers KV-cache / SDPA ops
import executorch.kernels.quantized                      # registers 4-bit kernels
from executorch.extension.llm.runner import TextLLMRunner, GenerationConfig

def chat(system, user):
    return ("<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n" + system +
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n" + user +
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n")

SMS_SYSTEM = ("You are a mobile security assistant. A scam detector has already analysed an SMS. "
              "In 2 or 3 short sentences, explain to a non-technical person why the message is or is not "
              "risky, then give one clear action starting with 'Action:'. Do not invent facts.")
SMS_USER = ('SMS: "Dear customer, your SBI account KYC is pending and will be blocked today. Update now: http://sbi-kyc-update.xyz"\n'
            "Detector verdict: Scam (scam probability 93%)\n"
            "Words that raised the score: <link>, account, customer, dear customer, today\n"
            "Red flags found: Contains a link, Mentions KYC / account block, Urgent language\n"
            "Explain and recommend an action.")
NET_SYSTEM = ("You are a network security assistant on a home or small-office router. A detector has already "
              "classified one network flow. In 2 or 3 short sentences, explain to a non-expert what the verdict "
              "means and which measurements look unusual, then give one clear step starting with 'Action:'. "
              "Do not invent facts.")
NET_USER = ("Destination port: 3645\nDetector verdict: Bots (confidence 100%)\n"
            "Most unusual measurements compared with normal traffic: Bwd Packets/s = 37,037 (normal is about 48); "
            "ACK Flag Count = 1 (normal is about 0); Packet Length Std = 0 (normal is about 24)\n"
            "Explain and recommend an action.")

t0 = time.time(); runner = TextLLMRunner("llama3_2_1b.pte", TOKENIZER); print(f"Loaded in {time.time()-t0:.1f}s")
for name, (s, u) in {"SMS": (SMS_SYSTEM, SMS_USER), "Network": (NET_SYSTEM, NET_USER)}.items():
    runner.reset(); pieces, stats = [], {}
    def on_stats(st):
        stats["tok"] = st.num_generated_tokens
        stats["tps"] = st.num_generated_tokens / max((st.inference_end_ms - st.prompt_eval_end_ms) / 1000, 1e-6)
    t0 = time.time()
    runner.generate(chat(s, u), GenerationConfig(echo=False, max_new_tokens=96, temperature=0.0),
                    token_callback=pieces.append, stats_callback=on_stats)
    print(f"\n=== {name} ({time.time()-t0:.1f}s, {stats.get('tok')} tokens, {stats.get('tps', 0):.1f} tokens/s on Colab CPU)")
    print("".join(pieces).split("<|eot_id|>")[0].strip())

[llm_runner_helper.cpp:72] Loaded TikToken tokenizer
[llm_runner_helper.cpp:258] Reading metadata from model
[llm_runner_helper.cpp:137] Metadata: use_kv_cache = 1
[llm_runner_helper.cpp:137] Metadata: get_max_context_len = 1024
[llm_runner_helper.cpp:137] Metadata: get_max_seq_len = 1024
[llm_runner_helper.cpp:137] Metadata: use_sdpa_with_kv_cache = 1
[llm_runner_helper.cpp:137] Metadata: enable_dynamic_shape = 1
[llm_runner_helper.cpp:179] eos_id = 128009
[llm_runner_helper.cpp:179] eos_id = 128001
[llm_runner_helper.cpp:294] Using method: forward


Loaded in 0.1s

=== SMS (21.6s, 95 tokens, 9.8 tokens/s on Colab CPU)
Here's the explanation:

The message is not a scam, but rather a phishing attempt. The words that raised the score are: <link>, account, customer, dear customer, today. These words are commonly used in phishing emails to create a sense of urgency and to make the recipient feel like they need to take immediate action.

The red flags found are: Contains a link, Mentions KYC / account block, Urgent language. These red flags indicate that the message is

=== Network (18.2s, 95 tokens, 8.8 tokens/s on Colab CPU)
Here's a breakdown of the verdict and unusual measurements:

The detector has classified the network flow as "bots", which means it's likely that the flow is coming from a botnet (a network of infected computers). The verdict of 100% confidence is high, indicating that the detector is very sure that the flow is likely to be from a botnet.

The unusual measurements are:

* Bwd Packets/s: 37,037 (normal is about 48)

## 6. Save the files
The `.pte` is about 1 GB, so a browser download can be slow. We copy both files to your **Google Drive** (folder `llama_pte`) and also try a direct download.

In [12]:
tok_name = "tokenizer.model" if TOKENIZER.endswith(".model") else "tokenizer.json"
shutil.copy(TOKENIZER, tok_name)
try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/llama_pte", exist_ok=True)
    for f in ["llama3_2_1b.pte", tok_name]:
        shutil.copy(f, "/content/drive/MyDrive/llama_pte/")
    print("Copied to Google Drive: My Drive/llama_pte/")
    files.download(tok_name)
    files.download("llama3_2_1b.pte")
except ImportError:
    print("Not in Colab: files are in the current folder")

Mounted at /content/drive
Copied to Google Drive: My Drive/llama_pte/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Put it in the app
1. Copy `llama3_2_1b.pte` and `tokenizer.model` into `sms-scam-detector/models/`.
   (With plan B the tokenizer is `tokenizer.json`. Before starting the app, run `set LLAMA_TOKENIZER_PATH=models\tokenizer.json` on Windows or `export LLAMA_TOKENIZER_PATH=models/tokenizer.json` on Linux/Mac.)
2. `pip install -r requirements.txt -r requirements-llm.txt`
3. `streamlit run app/app.py`. The sidebar turns green: *"Loaded llama3_2_1b.pte…"*. Both the SMS page and the Network Threats page now show Llama's explanations.
4. Measure: `python src/benchmark.py --llm` and put the numbers in the report.

⚠️ Don't upload the `.pte` or your token to GitHub (`.gitignore` already blocks `*.pte`).